# Preliminary Transformers

**Purpose.** Transformer architecture experiments from the preliminary phase.

**Thesis reference.** §5.1 Preliminary Work

**Inputs / outputs.** See the data-layout table in the repository README. Paths near the top of the notebook (e.g. `PROFILE_DIR`, `IMAGE_ROOT`, `LABELS`) are relative to the repository root and assume the folder layout described there.

---

In [1]:
import os
import random
import mlflow
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import timm
import numpy as np
import xgboost as xgb
from xgboost.callback import EarlyStopping as XGBoostEarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

In [2]:
diab_wo_ckd = ["P0067", "P0187", "P0231", "P0486", "P0488", "P0508", "P0510", "P0511", "P0556", "P0603", "P0609",
               "P0744", "P0767", "P0778", "P0784", "P0800", "P0832", "P0834", "P0858", "P0863", "P0869", "P0876",
               "P0895", "P0934", "P0938", "P0950", "P0960", "P0961", "P0963", "P0964", "P0967", "P0969", "P0976",
               "P0981", "P1000", "P1004", "P1009", "P1010", "P1011", "P1015", "P1024", "P1030", "P1032", "P1033"]

diab_ckd1 = ["P0045", "P0098", "P0110", "P0210", "P0279", "P0309", "P0316", "P0332", "P0375", "P0470", "P0598", "P0605",
             "P0690", "P0753", "P0783", "P0819", "P0909", "P0922", "P0925", "P0948", "P0965", "P0966", "P0968", "P0974",
             "P1007", "P1008", "P1012", "P1018", "P1021", "P1025", "P1028", "P1029"]

diab_ckd2 = ["P0004", "P0010", "P0013", "P0069", "P0070", "P0079", "P0080", "P0083", "P0103", "P0143", "P0160", "P0161",
             "P0175", "P0181", "P0199", "P0203", "P0205", "P0213", "P0214", "P0223", "P0226", "P0238", "P0240", "P0246",
             "P0260", "P0261", "P0285", "P0318", "P0327", "P0330", "P0341", "P0358", "P0362", "P0364", "P0369", "P0383",
             "P0391", "P0399", "P0411", "P0419", "P0447", "P0459", "P0463", "P0466", "P0480", "P0483", "P0490", "P0495",
             "P0497", "P0499", "P0506", "P0518", "P0558", "P0600", "P0611", "P0614", "P0618", "P0620", "P0624", "P0628",
             "P0631", "P0654", "P0662", "P0672", "P0679", "P0689", "P0699", "P0708", "P0711", "P0715", "P0729", "P0759",
             "P0775", "P0787", "P0795", "P0804", "P0822", "P0826", "P0830", "P0848", "P0859", "P0873", "P0882", "P0883",
             "P0884", "P0903", "P0911", "P0919", "P0932", "P0946", "P0970", "P0972", "P0975", "P0977", "P0994", "P0998",
             "P1006", "P1013", "P1016", "P1017", "P1022", "P1026"]

diab_ckd3 = ["P0009", "P0012", "P0101", "P0115", "P0121", "P0137", "P0138", "P0172", "P0179", "P0182", "P0184", "P0194",
             "P0202", "P0221", "P0242", "P0265", "P0266", "P0269", "P0288", "P0310", "P0317", "P0326", "P0328", "P0334",
             "P0337", "P0342", "P0360", "P0376", "P0378", "P0385", "P0402", "P0404", "P0413", "P0414", "P0416", "P0425",
             "P0441", "P0452", "P0465", "P0476", "P0477", "P0485", "P0515", "P0523", "P0536", "P0545", "P0547", "P0550",
             "P0551", "P0568", "P0571", "P0572", "P0577", "P0602", "P0612", "P0621", "P0622", "P0634", "P0636", "P0645",
             "P0647", "P0660", "P0661", "P0663", "P0665", "P0667", "P0670", "P0680", "P0691", "P0692", "P0701", "P0702",
             "P0709", "P0748", "P0752", "P0821", "P0825", "P0843", "P0868", "P0880", "P0929", "P0973", "P1014", "P1027"]

diab_ckd4 = ["P0118", "P0222", "P0361", "P0445", "P0451", "P0656", "P0657", "P0760"]

diabetes_patients_non_ckd = ['P0067', 'P0187', 'P0231', 'P0486', 'P0488', 'P0508', 'P0510', 'P0511', 'P0556', 'P0603', 'P0609',
                     'P0744', 'P0767', 'P0778', 'P0784', 'P0800', 'P0832', 'P0834', 'P0858', 'P0863', 'P0869', 'P0876',
                     'P0895', 'P0934', 'P0938', 'P0950', 'P0960', 'P0961', 'P0963', 'P0964', 'P0967', 'P0969', 'P0976',
                     'P0981', 'P1000', 'P1004', 'P1009', 'P1010', 'P1011', 'P1015', 'P1024', 'P1030', 'P1032', 'P1033']

diabetes_patients_ckd = ['P0009', 'P0012', 'P0101', 'P0115', 'P0121', 'P0137', 'P0138', 'P0172', 'P0179', 'P0182',
                         'P0184', 'P0194', 'P0202', 'P0221', 'P0242', 'P0265', 'P0266', 'P0269', 'P0288', 'P0310',
                         'P0317', 'P0326', 'P0328', 'P0334', 'P0337', 'P0342', 'P0360', 'P0376', 'P0378', 'P0385',
                         'P0402', 'P0404', 'P0413', 'P0414', 'P0416', 'P0425', 'P0441', 'P0452', 'P0465', 'P0476',
                         'P0477', 'P0485', 'P0515', 'P0523', 'P0536', 'P0545', 'P0547', 'P0550', 'P0551', 'P0568',
                         'P0571', 'P0572', 'P0577', 'P0602', 'P0612', 'P0621', 'P0622', 'P0634', 'P0636', 'P0645',
                         'P0647', 'P0660', 'P0661', 'P0663', 'P0665', 'P0667', 'P0670', 'P0680', 'P0691', 'P0692',
                         'P0701', 'P0702', 'P0709', 'P0748', 'P0752', 'P0821', 'P0825', 'P0843', 'P0868', 'P0880',
                         'P0929', 'P0973', 'P1014', 'P1027', 'P0118', 'P0222', 'P0361', 'P0445', 'P0451', 'P0656',
                         'P0657', 'P0760']

non_diabetes_patients_non_ckd = ["P0038", "P0047", "P0048", "P0062", "P0126", "P0140", "P0157", "P0159", "P0166", "P0248",
                         "P0252", "P0254", "P0267", "P0290", "P0304", "P0340", "P0392", "P0396", "P0421", "P0450",
                         "P0479", "P0484", "P0509", "P0512", "P0513", "P0526", "P0546", "P0626", "P0648", "P0735",
                         "P0745", "P0758", "P0769", "P0771", "P0782", "P0786", "P0794", "P0806", "P0809", "P0813",
                         "P0823", "P0831", "P0835", "P0853", "P0861", "P0864", "P0872", "P0877", "P0886", "P0887",
                         "P0890", "P0896", "P0901", "P0904", "P0905", "P0907", "P0924", "P0926", "P0940", "P0945",
                         "P0947", "P1001"]

non_diabetes_patients_ckd = ["P0005", "P0009", "P0012", "P0018", "P0022", "P0052", "P0055", "P0068", "P0085", "P0092",
                             "P0099", "P0101", "P0115", "P0120", "P0121", "P0124", "P0125", "P0133", "P0135", "P0137",
                             "P0138", "P0153", "P0155", "P0167", "P0168", "P0172", "P0179", "P0180", "P0182", "P0184",
                             "P0185", "P0194", "P0196", "P0201", "P0202", "P0211", "P0221", "P0225", "P0236", "P0237",
                             "P0239", "P0242", "P0247", "P0265", "P0266", "P0269", "P0276", "P0288", "P0300", "P0310",
                             "P0312", "P0317", "P0319", "P0325", "P0326", "P0328", "P0334", "P0337", "P0342", "P0345",
                             "P0354", "P0357", "P0360", "P0371", "P0376", "P0377", "P0378", "P0385", "P0397", "P0398",
                             "P0402", "P0404", "P0407", "P0412", "P0413", "P0414", "P0415", "P0416", "P0418", "P0425",
                             "P0429", "P0437", "P0440", "P0441", "P0446", "P0452", "P0465", "P0475", "P0476", "P0477",
                             "P0478", "P0485", "P0487", "P0491", "P0496", "P0514", "P0515", "P0523", "P0536", "P0537",
                             "P0539", "P0540", "P0541", "P0542", "P0545", "P0547", "P0550", "P0551", "P0553", "P0561",
                             "P0568", "P0571", "P0572", "P0574", "P0577", "P0586", "P0589", "P0595", "P0596", "P0602",
                             "P0612", "P0616", "P0621", "P0622", "P0634", "P0635", "P0636", "P0642", "P0645", "P0647",
                             "P0651", "P0653", "P0659", "P0660", "P0661", "P0663", "P0665", "P0667", "P0668", "P0670",
                             "P0671", "P0675", "P0680", "P0691", "P0692", "P0693", "P0698", "P0701", "P0702", "P0709",
                             "P0731", "P0732", "P0747", "P0748", "P0752", "P0774", "P0776", "P0779", "P0781", "P0789",
                             "P0790", "P0807", "P0821", "P0825", "P0833", "P0836", "P0843", "P0849", "P0868", "P0880",
                             "P0898", "P0906", "P0910", "P0929", "P0973", "P1014", "P1023", "P1027", "P0118", "P0222",
                             "P0333", "P0361", "P0381", "P0445", "P0451", "P0503", "P0516", "P0527", "P0656", "P0657",
                             "P0681", "P0751", "P0760", "P0597"]

In [3]:
label_dict = {}

for pid in non_diabetes_patients_non_ckd + diabetes_patients_non_ckd:
    label_dict[pid] = 0
for pid in non_diabetes_patients_ckd + diabetes_patients_ckd:
    label_dict[pid] = 1
    
num_ckd_positive = sum(1 for label in label_dict.values() if label == 1)
num_ckd_negative = sum(1 for label in label_dict.values() if label == 0)
print(f"  CKD Positive (Label 1): {num_ckd_positive} patients")
print(f"  CKD Negative (Label 0): {num_ckd_negative} patients")

  CKD Positive (Label 1): 194 patients
  CKD Negative (Label 0): 106 patients


In [4]:
DATA_ROOT = "../ahen_data/filtered_tsv_2"
IMAGE_ROOT = "images"

IMAGE_TYPES = {
    # "grayscale_heatmap": os.path.join(IMAGE_ROOT, "grayscale_heatmap"),
    # "pseudocolor_heatmap": os.path.join(IMAGE_ROOT, "pseudocolor_heatmap"),
    # "rgb_mz_split": os.path.join(IMAGE_ROOT, "rgb_mz_split"),
    # "lineplot_intensity": os.path.join(IMAGE_ROOT, "lineplot_intensity"),
    "scatterplot": os.path.join(IMAGE_ROOT, "scatterplot"),
    "tic_plot": os.path.join(IMAGE_ROOT, "tic_plot"),
    "recurrence_plot": os.path.join(IMAGE_ROOT, "recurrence_plot"),
}

In [5]:
NGROK_FORWARDING_URL = "https://b172-77-28-146-225.ngrok-free.app"  
MLFLOW_TRACKING_URI = NGROK_FORWARDING_URL

print(f"NGROK_FORWARDING_URL: {NGROK_FORWARDING_URL}")
print(f"MLFLOW_TRACKING_URI set to: {MLFLOW_TRACKING_URI}")


try:
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    print(f"Successfully set tracking URI. mlflow.get_tracking_uri() reports: {mlflow.get_tracking_uri()}")
except Exception as e:
    print(f"Error setting tracking URI: {e}")
    raise

NGROK_FORWARDING_URL: https://b172-77-28-146-225.ngrok-free.app
MLFLOW_TRACKING_URI set to: https://b172-77-28-146-225.ngrok-free.app
Successfully set tracking URI. mlflow.get_tracking_uri() reports: https://b172-77-28-146-225.ngrok-free.app


In [6]:
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5])
])

In [7]:
class ImageDataset(Dataset):
    def __init__(self, patient_ids, label_dict, image_type, transform):
        self.patient_ids = patient_ids
        self.labels = [label_dict[pid] for pid in patient_ids]
        self.image_dir = IMAGE_TYPES[image_type]
        self.transform = transform

    def __len__(self):
        return len(self.patient_ids)

    def __getitem__(self, idx):
        pid = self.patient_ids[idx]
        img_path = os.path.join(self.image_dir, f"{pid}.png")
        img = Image.open(img_path).convert("RGB")
        return self.transform(img), self.labels[idx]

In [8]:
mlflow.set_experiment("CKD_transformer_swin_large_patch4_window7_224")

2025/06/23 08:06:42 INFO mlflow.tracking.fluent: Experiment with name 'CKD_transformer_swin_large_patch4_window7_224' does not exist. Creating a new experiment.


<Experiment: artifact_location='/mlflow/artifact_store/938885747210740134', creation_time=1750666005288, experiment_id='938885747210740134', last_update_time=1750666005288, lifecycle_stage='active', name='CKD_transformer_swin_large_patch4_window7_224', tags={}>

In [9]:
MODEL_ARCHITECTURES = [
    # 'vit_base_patch16_224', # ViT-Base DONE
    # 'vit_large_patch16_224', # ViT-Large DONE BATCH_SIZE = 16
    # 'deit_base_distilled_patch16_224', # DeiT DONE
    # 'swin_base_patch4_window7_224', # Swin Transformer DONE BATCH_SIZE = 16
    ### 'swin_large_patch4_window7_224', # Swin-Large UNDONE 
    # 'beit_base_patch16_224', # BEiT DONE
    # 'levit_256', # LeViT DONE
    # 'coat_lite_small', # CoaT DONE BATCH_SIZE = [16, 32]
]

In [10]:
NUM_WORKERS = 0
# LEARNING_RATES = [1e-3, 1e-4, 1e-5] # [0.001, 0.0001, 0.00001]
# LEARNING_RATES = [1e-4, 1e-5] # [0.001, 0.0001, 0.00001]
LEARNING_RATES = [1e-4, 5e-5, 1e-5]
# BATCH_SIZES = [16, 32, 64]
BATCH_SIZES = [16]

In [11]:
def train_transformer(model_name, train_loader, val_loader, device, epochs=5, lr=1e-5, T_max=10, patience=3, max_grad_norm=1.0):
    model = timm.create_model(model_name, pretrained=True, num_classes=2)
    model.to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=T_max)

    best_auc = 0.0
    wait = 0
    best_model_state = None

    # with mlflow.start_run(run_name=f"transformer_{model_name}"):
    #     mlflow.log_params({
    #         'model': model_name,
    #         'epochs': epochs,
    #         'lr': lr,
    #         'optimizer': 'AdamW',
    #         'scheduler': 'CosineAnnealingLR',
    #         'label_smoothing': 0.1
    #     })
    for epoch in range(epochs):
            # training
        model.train()
        train_loss = 0.0
        all_preds, all_labels = [], []
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model(imgs)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()
            train_loss += loss.item() * imgs.size(0)
            probs = torch.softmax(logits, dim=1)[:,1].cpu().detach().numpy()
            preds = (probs > 0.5).astype(int)
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().detach().numpy())
        scheduler.step()
        # calculate train metrics
        train_loss /= len(train_loader.dataset)
        train_acc = accuracy_score(all_labels, all_preds)
        train_precision = precision_score(all_labels, all_preds, zero_division=0)
        train_recall = recall_score(all_labels, all_preds, zero_division=0)
        train_f1 = f1_score(all_labels, all_preds, zero_division=0)

        # validation
        model.eval()
        val_loss = 0.0
        val_preds, val_labels, val_probs = [], [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                logits = model(imgs)
                loss = criterion(logits, labels)
                val_loss += loss.item() * imgs.size(0)
                probs = torch.softmax(logits, dim=1)[:,1].cpu().detach().numpy()
                preds = (probs > 0.5).astype(int)
                val_probs.extend(probs)
                val_preds.extend(preds)
                val_labels.extend(labels.cpu().detach().numpy())
        val_loss /= len(val_loader.dataset)
        val_acc = accuracy_score(val_labels, val_preds)
        val_precision = precision_score(val_labels, val_preds, zero_division=0)
        val_recall = recall_score(val_labels, val_preds, zero_division=0)
        val_f1 = f1_score(val_labels, val_preds, zero_division=0)
        try:
            val_auc = roc_auc_score(val_labels, val_probs)
        except ValueError:
            val_auc = float('nan')

        # early stopping on AUC
        if val_auc > best_auc:
            best_auc = val_auc
            best_model_state = model.state_dict()
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break
        
        # log all metrics
        metrics = {
            "train_loss": train_loss,
            "train_acc": train_acc,
            "train_precision": train_precision,
            "train_recall": train_recall,
            "train_f1": train_f1,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "val_precision": val_precision,
            "val_recall": val_recall,
            "val_f1": val_f1,
            "val_auc": val_auc
        }
        
        # best model
    if best_model_state:
        model.load_state_dict(best_model_state)
    mlflow.log_metric('best_val_auc', best_auc)

    return model, metrics


In [12]:
# extract embeddings (feature vectors) from transformer models for each image 
def extract_embeddings(model, loader, device):
    model.eval()
    model.to(device)
    
    all_embeddings = []
    all_labels = []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            
            feats = model.forward_features(imgs) # shape (batch_size, num_patches, embed_dim)

            # get the single feature vector that the head uses
            if hasattr(model, 'fc_norm'):
                # for viT it's the CLS token/index 0
                embedding = model.fc_norm(feats[:, 0])
            elif hasattr(model, 'norm'):
                 # fallback for other models
                embedding = model.norm(feats)[:, 0]
            else:
                embedding = feats.mean(dim=1)
             # some models return (cls, dist)
            feats = model.forward_features(imgs)
            if isinstance(feats, tuple):
                 # we only need the CLS token embedding
                embedding = feats[0]                 
            else:
                if hasattr(model, 'fc_norm'):
                    embedding = model.fc_norm(feats[:, 0])
                elif hasattr(model, 'norm'):
                    embedding = model.norm(feats)[:, 0]
                else:
                    embedding = feats.mean(dim=1)
            all_embeddings.append(embedding.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    # join batches into a single 2D array
    final_embeddings = np.concatenate(all_embeddings, axis=0)
    if final_embeddings.ndim == 3:
        final_embeddings = final_embeddings.mean(axis=1)
    final_labels = np.concatenate(all_labels, axis=0)
    
    return final_embeddings, final_labels

In [13]:
def run_experiment(test_size=0.2, epochs=10):
    image_types = list(IMAGE_TYPES.keys())
    all_pids = list(label_dict.keys())
    all_labels = [label_dict[pid] for pid in all_pids]
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    for image_type in image_types:
        all_pids = [pid for pid in label_dict.keys() if os.path.exists(os.path.join(IMAGE_TYPES[image_type], f"{pid}.png"))]
        all_labels = [label_dict[pid] for pid in all_pids]
        if not all_pids:
            print(f"No valid images for image_type={image_type}.")
            continue

        # presplit once per img/vis type for consistent splits
        train_pids, val_pids = train_test_split(all_pids, test_size=test_size, stratify=all_labels, random_state=42)

        for lr in LEARNING_RATES:
            for batch_size in BATCH_SIZES:
                run_name = f"{image_type}_{model_name}_lr{lr}_bs{bs}"
                with mlflow.start_run(run_name=run_name):
                    mlflow.log_params({'image_type': image_type,
                                      'learning_rate': lr,
                                      'batch_size': batch_size,
                                      'epochs': epochs})

                    # prep data loaders
                    train_loader = DataLoader(ImageDataset(train_pids, label_dict, image_type, train_transform), batch_size=batch_size, shuffle=True, num_workers=NUM_WORKERS)
                    val_loader = DataLoader(ImageDataset(val_pids, label_dict, image_type, val_transform), batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS)

                    # train transformers
                    trained_models = {}
                    for name in MODEL_ARCHITECTURES:
                        model = train_transformer(name, train_loader, val_loader, device, epochs, lr, T_max=epochs)
                        trained_models[name] = model

                    # XGBoost on embeddings
                    for name, model in trained_models.items():
                        with mlflow.start_run(run_name=f"XGB_{image_type}_{name}", nested=True):
                            mlflow.log_params({'image_type': image_type,
                                              'transformer': name,
                                              'classifier': 'XGBClassifier'})
                            tr_emb, tr_lab = extract_embeddings(model, train_loader, device)
                            val_emb, val_lab = extract_embeddings(model, val_loader, device)

                            clf = xgb.XGBClassifier(eval_metric='auc', early_stopping_rounds = 10)
                            clf.fit(tr_emb, 
                                    tr_lab, 
                                    eval_set=[(val_emb, val_lab)], 
                                    verbose=False)
                            preds = clf.predict_proba(val_emb)[:,1]
                            auc = roc_auc_score(val_lab, preds)
                            acc = accuracy_score(val_lab, preds>0.5)
                            mlflow.log_metrics({'xgb_val_acc': acc, 'xgb_val_auc': auc})
                            print(f"{run_name} - {name}: acc={acc:.4f}, auc={auc:.4f}")

In [14]:
def run_experiment(test_size=0.2, epochs=10):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    for image_type in IMAGE_TYPES:
        all_pids = [pid for pid in label_dict if os.path.exists(os.path.join(IMAGE_TYPES[image_type], f"{pid}.png"))]
        all_labels = [label_dict[pid] for pid in all_pids]

        if not all_pids:
            continue

        # presplit once per img/vis type for consistent splits
        train_pids, val_pids = train_test_split(all_pids, test_size=test_size, stratify=all_labels, random_state=42)

        for model_name in MODEL_ARCHITECTURES:
            for lr in LEARNING_RATES:
                for bs in BATCH_SIZES:
                    run_name = f"{image_type}_{model_name}_lr{lr}_bs{bs}"
                    with mlflow.start_run(run_name=run_name):
                        mlflow.log_params({
                            'image_type': image_type,
                            'model': model_name,
                            'learning_rate': lr,
                            'batch_size': bs,
                            'epochs': epochs,
                            'optimizer': 'AdamW',
                            'scheduler': 'CosineAnnealingLR',
                            'classifier': 'XGBClassifier'
                        })

                        # prep data loaders
                        train_loader = DataLoader(ImageDataset(train_pids, label_dict, image_type, train_transform), batch_size=bs, shuffle=True, num_workers=NUM_WORKERS)
                        val_loader = DataLoader(ImageDataset(val_pids, label_dict, image_type, val_transform), batch_size=bs, shuffle=False, num_workers=NUM_WORKERS)

                        # train transformers
                        model, val_metrics = train_transformer(model_name, train_loader, val_loader, device, epochs, lr, T_max=epochs)
                        mlflow.log_metrics(val_metrics)

                        # extract embeddings
                        tr_emb, tr_lab = extract_embeddings(model, train_loader, device)
                        val_emb, val_lab = extract_embeddings(model, val_loader, device)

                        # XGBoost on embeddings
                        clf = xgb.XGBClassifier(eval_metric='auc', early_stopping_rounds=10)
                        clf.fit(tr_emb, tr_lab, eval_set=[(val_emb, val_lab)], verbose=False)
                        preds = clf.predict_proba(val_emb)[:, 1]
                        xgb_auc = roc_auc_score(val_lab, preds)
                        xgb_acc = accuracy_score(val_lab, preds > 0.5)

                        mlflow.log_metrics({
                            'xgb_val_acc': xgb_acc,
                            'xgb_val_auc': xgb_auc
                        })

                        print(f"{run_name} - val_auc={val_metrics['val_auc']:.3f}, xgb_auc={xgb_auc:.3f}")

In [ ]:
run_experiment()